In [1]:
import numpy
import os
import pandas as pd

## 1. Processing customer data from a CSV file for a retail company to ensure clean, unique records.
### Tasks involved: Deduplication, Cleaning

In [2]:
transaction_data = pd.read_csv('transaction_data.csv')
customer_data = pd.read_csv('us_customer_data.csv')

In [3]:
print('transaction_data count',transaction_data.shape)
print('customer_data count', customer_data.shape)

transaction_data count (1000, 7)
customer_data count (1000, 7)


In [4]:

pd.set_option('display.max_columns', None) 
def data_overview(df, head=5):
    print("DATA TYPES".center(125,'-'))
    print(df.dtypes.value_counts())
    
    print(" MISSING VALUES ".center(125,'-'))
    print(df.isnull().sum()[df.isnull().sum()>0].sort_values(ascending = False))
    
    print(" DUPLICATED VALUES ".center(125,'-'))
    print(df.duplicated().sum())
    
    print(" STATISTICS OF DATA ".center(125,'-'))
    print(df.describe(include="all"))
    
    print("DATA INFO".center(125,'-'))
    print(df.info())
    
    
data_overview(transaction_data)

----------------------------------------------------------DATA TYPES---------------------------------------------------------
object     4
int64      2
float64    1
Name: count, dtype: int64
------------------------------------------------------- MISSING VALUES ------------------------------------------------------
amount    50
dtype: int64
----------------------------------------------------- DUPLICATED VALUES -----------------------------------------------------
0
----------------------------------------------------- STATISTICS OF DATA ----------------------------------------------------
        transaction_id  customer_id       amount     transaction_date  \
count      1000.000000  1000.000000   950.000000                 1000   
unique             NaN          NaN          NaN                 1000   
top                NaN          NaN          NaN  2025-03-10 01:20:54   
freq               NaN          NaN          NaN                    1   
mean        500.500000   489.319000  2

In [5]:
print("Missing customer data in each column after cleaning customerID :\n",customer_data.isnull().sum())
customer_data.dropna(subset=["customer_id"],axis = 0 , inplace = True)



Missing customer data in each column after cleaning customerID :
 customer_id           0
name                  0
email                50
phone                50
address               0
registration_date     0
loyalty_status        0
dtype: int64


In [6]:
print("Missing customer data in each column after cleaning customerID :\n",customer_data.isnull().sum())



Missing customer data in each column after cleaning customerID :
 customer_id           0
name                  0
email                50
phone                50
address               0
registration_date     0
loyalty_status        0
dtype: int64


In [7]:
print('before cleaning' , customer_data.email.duplicated().sum())
customer_data = customer_data.drop_duplicates(keep='first')
print('after cleaning' , customer_data.duplicated().sum())

before cleaning 139
after cleaning 0


In [8]:
customer_data.shape

(1000, 7)

## 2. Standardizing contact details in an Excel file for a marketing campaign using  customer data
### Tasks involved: Cleaning, Mapping

In [9]:
# remove whitespace 
import pandas as pd
import re

def clean_spaces(x):
    if pd.isna(x):
        return ""
    return re.sub(r"\s+", " ", str(x).strip())


In [10]:

for col in ['email','phone','name']:
    if customer_data[col].dtype == "object":
     customer_data[col] = customer_data[col].apply(clean_spaces)


In [11]:
def proper_case_name(name):
    name = clean_spaces(name)
    return name.title() if name else ""

customer_data["name"] = customer_data["name"].apply(proper_case_name)


In [12]:
# fix email error
EMAIL_REGEX = r"^[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}$"

def clean_email(email):
    email = clean_spaces(email).lower()
    return email

def email_valid(email):
    return bool(re.match(EMAIL_REGEX, email)) if email else False

customer_data["email_clean"] = customer_data["email"].apply(clean_email)
customer_data["email_valid"] = customer_data["email_clean"].apply(email_valid)

# Optional: blank out invalid emails
customer_data.loc[~customer_data["email_valid"], "email_clean"] = ""


In [14]:
# standardize phone number
import re
# formats
# (716)547-9134x123
# 001-716-547-9134x123
# 716.547.9134x123
# rest are junk
# output: us standar: phoneno: +1-716- 547-9134 phoneext: digits if present


s = str('(716)547-9134').strip()




In [18]:
df = customer_data

In [19]:
import pandas as pd
import re

def clean_us_phone_with_ext(raw):
    """
    Returns (phone_e164, ext) for US numbers.
    phone_e164 format: +1-AAA-BBB-CCCC
    ext is digits only (string) or None
    """
    if pd.isna(raw):
        return (None, None)

    s = str(raw).strip()
    if s == "":
        return (None, None)

    # --- 1) Extract extension (x123, ext 123, extension 123) ---
    ext = None
    m = re.search(r'(?:ext\.?|extension|x)\s*(\d+)\s*$', s, flags=re.IGNORECASE)
    if m:
        ext = m.group(1)
        s = s[:m.start()].strip()  # remove extension part from main number

    # --- 2) Keep digits only from the remaining number ---
    digits = re.sub(r"\D", "", s)

    # --- 3) Remove leading international prefix "001" (common in your data) ---
    if digits.startswith("001"):
        digits = digits[3:]

    # --- 4) Normalize to US 10-digit ---
    # If 11 digits and starts with 1 -> drop country code
    if len(digits) == 11 and digits.startswith("1"):
        digits = digits[1:]

    # If not exactly 10 digits now, it's invalid for US standardization
    if len(digits) != 10:
        return (None, ext)  # keep ext if it existed, but number invalid

    phone_e164 = f"+1-{digits[:3]}-{digits[3:6]}-{digits[6:]}"
    return (phone_e164, ext)

# Apply to your dataframe
df[["phone_e164", "phone_ext"]] = df["phone"].apply(
    lambda x: pd.Series(clean_us_phone_with_ext(x))
)

# (Optional) Replace original phone with standardized phone
df["phone"] = df["phone_e164"]
df.drop(columns=["phone_e164"], inplace=True)

df[["phone", "phone_ext"]].head(20)


,phone,phone_ext
0,None,None
1,+1-759-518-8536,738
2,+1-323-525-3094,96062
3,+1-947-633-4224,07930
4,+1-869-650-5682,8385
5,+1-971-475-6369,8486
6,+1-418-931-4146,588
7,+1-593-948-3872,None
8,+1-710-854-4550,None
9,+1-920-793-4515,302


In [22]:
transaction_data.head()

,transaction_id,customer_id,amount,transaction_date,product_category,payment_method,store_location
0,1,565,2992.47,2025-03-10 01:20:54,Sports,Debit Card,New York
1,2,323,2041.87,2025-01-02 15:24:19,Clothing,Cash,New York
2,3,398,107.35,2025-02-16 03:49:01,Beauty,Debit Card,Online
3,4,19,NaN,2025-04-30 15:26:23,Sports,Debit Card,Los Angeles
4,5,547,3063.28,2025-06-14 04:28:53,Clothing,PayPal,Los Angeles


In [23]:
print(transaction_data.info())
print(transaction_data.isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   transaction_id    1000 non-null   int64  
 1   customer_id       1000 non-null   int64  
 2   amount            950 non-null    float64
 3   transaction_date  1000 non-null   object 
 4   product_category  1000 non-null   object 
 5   payment_method    1000 non-null   object 
 6   store_location    1000 non-null   object 
dtypes: float64(1), int64(2), object(4)
memory usage: 54.8+ KB
None
transaction_id       0
customer_id          0
amount              50
transaction_date     0
product_category     0
payment_method       0
store_location       0
dtype: int64


In [24]:
transaction_data = transaction_data.dropna(subset=["amount"])


In [25]:
transaction_data["amount"] = pd.to_numeric(
    transaction_data["amount"], errors="coerce"
)


In [26]:
transaction_data["transaction_date"] = pd.to_datetime(
    transaction_data["transaction_date"], errors="coerce"
)


In [27]:
print(transaction_data.info())
print(transaction_data.isna().sum())

<class 'pandas.core.frame.DataFrame'>
Index: 950 entries, 0 to 999
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   transaction_id    950 non-null    int64         
 1   customer_id       950 non-null    int64         
 2   amount            950 non-null    float64       
 3   transaction_date  950 non-null    datetime64[ns]
 4   product_category  950 non-null    object        
 5   payment_method    950 non-null    object        
 6   store_location    950 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(3)
memory usage: 59.4+ KB
None
transaction_id      0
customer_id         0
amount              0
transaction_date    0
product_category    0
payment_method      0
store_location      0
dtype: int64


In [29]:

mean_amount = transaction_data["amount"].mean()
transaction_data["mean_amount"] = mean_amount   # same value in every row (useful for debugging)

# 3) Method 1: fixed threshold > 1000
transaction_data["is_high_value_1000"] = transaction_data["amount"].gt(1000)

# 4) Method 2: above mean
transaction_data["is_high_value_above_mean"] = transaction_data["amount"].gt(mean_amount)

transaction_data.head()


,transaction_id,customer_id,amount,transaction_date,product_category,payment_method,store_location,mean_amount,is_high_value_1000,is_high_value_above_mean
0,1,565,2992.47,2025-03-10 01:20:54,Sports,Debit Card,New York,2532.550126,True,True
1,2,323,2041.87,2025-01-02 15:24:19,Clothing,Cash,New York,2532.550126,True,False
2,3,398,107.35,2025-02-16 03:49:01,Beauty,Debit Card,Online,2532.550126,False,False
4,5,547,3063.28,2025-06-14 04:28:53,Clothing,PayPal,Los Angeles,2532.550126,True,True
5,6,752,441.49,2025-03-28 06:15:55,Electronics,Cash,New York,2532.550126,False,False
